# TASK 3.7 — Write Final Output + Notify

Selects the canonical output schema from the re-routed DataFrame, writes to
`atims_prod.ml_output.non_norad_predictions`, and POSTs a completion webhook
to the FastAPI backend.

By the time we get here every record already carries `final_category`,
`taxability`, `Reverse_UseTax`, write-up and QA columns (the unification in
Task 3.3 means rule/RAG/ML rows share one schema — AUDIT B1/B2), so this task
is a pure projection + write.

In [ ]:
%run ./config_notebook

In [ ]:
from pyspark.sql.functions import col, lit, when, concat_ws

df = read_stage(spark, STAGE["reroute"])
# Normalize case so the canonical FINAL_COLS resolve exactly (no case-duplicate cols).
df = df.toDF(*[c.lower() for c in df.columns])

In [ ]:
# Derive provenance columns expected by the output schema.
df = (df
      .withColumn("rule_applied",
                  when(col("match_type") == "det", lit(True)).otherwise(lit(False)))
      .withColumn("components_used",
                  concat_ws("+",
                            when(col("match_type").isin("det", "rag"), lit("rules")),
                            when(col("match_type") == "ml", lit("classifier")),
                            when(col("ensemble_confidence") == "RESOLVE", lit("llm_resolve")),
                            lit("tax_lookup"), lit("tax_writer"), lit("qa"))))

if "reviewed_by" not in df.columns:
    df = df.withColumn("reviewed_by", lit(None).cast("string"))

In [ ]:
FINAL_COLS = [
    "invoice_id", "inv_line_number", "final_description",
    "vendor_name", "state", "amount", "final_category", "taxability",
    "use_tax_amount", "Reverse_UseTax",
    "confidence", "ensemble_confidence", "rule_applied", "match_type",
    "short_justification", "full_writeup", "audit_ready", "reviewer_note",
    "anomalies", "anomaly_score", "review_priority",
    "components_used", "reviewed_by", "processing_path",
]

# Add any missing optional columns so the projection never fails.
_have = {x.lower() for x in df.columns}
for c in FINAL_COLS:
    if c.lower() not in _have:
        df = df.withColumn(c, lit(None))

final = df.select(*FINAL_COLS)

(final.write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(TABLES["predictions_out"]))

total   = final.count()
flagged = final.filter(col("review_priority").isin("HIGH", "CRITICAL")).count()
print(f"[3.7] wrote {total:,} rows → {TABLES['predictions_out']}  (flagged: {flagged:,})")

### Notify FastAPI backend

In [ ]:
def _notify(month, records_processed, flagged_count):
    if not WEBHOOK_URL:
        print("[3.7] WEBHOOK_URL not set (Secret Scope) — skipping notify.")
        return
    import json, urllib.request
    payload = json.dumps({"month": month,
                          "records_processed": records_processed,
                          "flagged_count": flagged_count}).encode()
    req = urllib.request.Request(WEBHOOK_URL, data=payload,
                                 headers={"Content-Type": "application/json"})
    try:
        urllib.request.urlopen(req, timeout=30)
        print("[3.7] webhook POSTed to FastAPI.")
    except Exception as e:
        print(f"[3.7] webhook failed: {e}")


_notify(MONTH, total, flagged)
dbutils.notebook.exit("3.7 OK")